In [14]:
import pandas as pd
import seaborn as sns
import numpy as np
import tensorflow as tf
import tensorboard as tb
import matplotlib.pyplot as plt

from data_utils import load_clean

In [15]:
df = pd.read_csv('../dataset/Iisingintie_Acc1_m800_RDMC_2026-05-06_123750.txt', sep='\t')

df.head()

C:\Users\Nihar Buha\AppData\Local\Temp\ipykernel_20856\3505644561.py:1: DtypeWarning: Columns (18: posneg, 19: vaikeus, 20: muu) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../dataset/Iisingintie_Acc1_m800_RDMC_2026-05-06_123750.txt', sep='\t')


,Num,Timer(ms),Loc(m),Speed (m/s),East(m),North(m),CElev(m),Roll(deg),Pitch(deg),Yaw(deg),...,AccY,AccZ,GyrX,GyrY,GyrZ,RBCSV_80(cf),RBCSV_60(cf),posneg,vaikeus,muu
0,1,45476731,"3767,52","0,035","426290,475","7414101,054","107,049","1,6195","-1,7959","13,5969",...,"0,2478","9,84059","-0,00239","0,00271","-0,00435","0,87092","0,85334",0,0,0
1,2,"45476733,5","3767,52","0,035","426290,475","7414101,054","107,049","1,6197","-1,7954","13,5971",...,"0,22563","9,83939","-0,00394","0,00529","-0,00295","0,87092","0,85334",0,0,0
2,3,"45476728,5","3767,52","0,035","426290,475","7414101,054","107,048","1,6192","-1,7961","13,5968",...,"0,25581","9,83537","-0,00354","0,00251","-0,00404","0,87092","0,85334",0,0,0
3,4,45476736,"3767,52","0,035","426290,475","7414101,054","107,049","1,6196","-1,7955","13,5979",...,"0,22451","9,83531","-0,0054","0,00077","0,00153","0,87092","0,85334",0,0,0
4,5,45476726,"3767,52","0,035","426290,475","7414101,054","107,048","1,619","-1,7962","13,5968",...,"0,27895","9,82614","-0,00743","-0,0013","-0,00236","0,87092","0,85334",0,0,0


In [16]:
df.shape

(69125, 21)

In [17]:
df.dtypes

Num              int64
Timer(ms)          str
Loc(m)             str
Speed (m/s)        str
East(m)            str
North(m)           str
CElev(m)           str
Roll(deg)          str
Pitch(deg)         str
Yaw(deg)           str
AccX               str
AccY               str
AccZ               str
GyrX               str
GyrY               str
GyrZ               str
RBCSV_80(cf)       str
RBCSV_60(cf)       str
posneg          object
vaikeus         object
muu             object
dtype: object

In [18]:
df.describe()

,Num
count,69125.000000
mean,34563.000000
std,19954.813016
min,1.000000
25%,17282.000000
50%,34563.000000
75%,51844.000000
max,69125.000000


In [19]:
df.isna().sum()

Num             0
Timer(ms)       0
Loc(m)          0
Speed (m/s)     1
East(m)         0
North(m)        0
CElev(m)        0
Roll(deg)       0
Pitch(deg)      0
Yaw(deg)        0
AccX            0
AccY            0
AccZ            0
GyrX            0
GyrY            0
GyrZ            0
RBCSV_80(cf)    0
RBCSV_60(cf)    0
posneg          0
vaikeus         0
muu             0
dtype: int64

In [20]:
df = df.drop(columns=['Num'])

In [21]:
for col in df.columns:
        df[col] = pd.to_numeric(df[col].str.replace(",", "."), errors="coerce")

In [22]:
df['muu'].value_counts()


muu
0.000    48904
1.000    11468
3.000     2351
4.000     1094
5.000     1067
         ...  
0.569        1
0.274        1
0.225        1
0.077        1
0.028        1
Name: count, Length: 527, dtype: int64

In [23]:
# Find rows where muu is not a whole number
odd = df["muu"] % 1 != 0
print("Odd muu rows:", odd.sum())
print("Smallest / largest odd value:", df.loc[odd, "muu"].min(), df.loc[odd, "muu"].max())

Odd muu rows: 4241
Smallest / largest odd value: 0.001 4.94


In [24]:
# Check if posneg and vaikeus have odd values too

print("Odd posneg:", (df["posneg"] % 1 != 0).sum())
print("Odd vaikeus:", (df["vaikeus"] % 1 != 0).sum())

Odd posneg: 4093
Odd vaikeus: 4093


In [25]:
# Look at one odd row together with its neighbours
i = df.index[odd][0]
print(df.loc[i-3:i+25, ["Loc(m)", "posneg", "vaikeus", "muu"]])

        Loc(m)  posneg  vaikeus    muu
9048  3887.873     0.0      0.0  0.000
9049  3887.923     0.0      0.0  0.000
9050  3887.973     0.0      0.0  0.000
9051  3888.022     0.0      0.0  0.066
9052  3888.072     0.0      0.0  0.216
9053  3888.122     0.0      0.0  0.366
9054  3888.172     0.0      0.0  0.516
9055  3888.222     0.0      0.0  0.666
9056  3888.271     0.0      0.0  0.813
9057  3888.321     0.0      0.0  0.963
9058  3888.371     0.0      0.0  1.113
9059  3888.421     0.0      0.0  1.263
9060  3888.471     0.0      0.0  1.413
9061  3888.521     0.0      0.0  1.563
9062  3888.571     0.0      0.0  1.713
9063  3888.621     0.0      0.0  1.863
9064  3888.671     0.0      0.0  2.013
9065  3888.720     0.0      0.0  2.160
9066  3888.770     0.0      0.0  2.310
9067  3888.820     0.0      0.0  2.460
9068  3888.870     0.0      0.0  2.610
9069  3888.920     0.0      0.0  2.760
9070  3888.970     0.0      0.0  2.910
9071  3889.020     0.0      0.0  3.000
9072  3889.070     0.0   

In [26]:
from pathlib import Path

for f in sorted(Path("../dataset").glob("*.txt")):
    df = load_clean(f)
    # Check empty values for each file, as this one has two
    print(f"  empty values:   {df.isna().sum().sum()}")
    # Check how many times the location goes back, which is not expected
    # I came to know about this chatting with AI,
    # it told me that there can be also negative values in the location column, which is not expected.
    print(f"  Loc goes back:  {(df['Loc(m)'].diff() < 0).sum()} times")
    # Check how many times the speed is below 7 km/h, which is considered nearly stopped
    print(f"  nearly stopped: {(df['Speed_kmh'] < 7).sum()} rows")

..\dataset\Iisingintie_Acc1_m800_RDMC_2026-05-06_123750.txt: 69125 rows, fixed 652 blended rows
  empty values:   2
  Loc goes back:  0 times
  nearly stopped: 6022 rows
..\dataset\Iisingintie_Acc1_m801_RDMC_2026-05-06_124454.txt: 83597 rows, fixed 833 blended rows
  empty values:   2
  Loc goes back:  0 times
  nearly stopped: 4753 rows
..\dataset\Iisingintie_Acc1_m802_RDMC_2026-05-06_125211.txt: 122623 rows, fixed 1248 blended rows
  empty values:   2
  Loc goes back:  0 times
  nearly stopped: 3691 rows
..\dataset\Kaittiaisentie_Acc1_RDMC_2026-05-06_132147.txt: 114943 rows, fixed 1475 blended rows
  empty values:   2
  Loc goes back:  0 times
  nearly stopped: 1882 rows
..\dataset\sinetänsalmentie_Acc1_m792_RDMC_2026-05-06_112539.txt: 9599 rows, fixed 84 blended rows
  empty values:   2
  Loc goes back:  0 times
  nearly stopped: 960 rows
..\dataset\sinetänsalmentie_Acc1_m793_RDMC_2026-05-06_112757.txt: 12985 rows, fixed 73 blended rows
  empty values:   2
  Loc goes back:  0 times
